In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm, ttest_ind
from sklearn.metrics import classification_report, confusion_matrix, make_scorer, matthews_corrcoef, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from statsmodels.stats.multitest import multipletests

DATA_DIR_CANDIDATES = (Path("acp740"), Path("acp_740"), Path("."))
DATA_DIR = next(
    (path for path in DATA_DIR_CANDIDATES if (path / "acp740_dataset_with_features.csv").is_file()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Could not locate the acp740 data directory.")
DATA_PATH = DATA_DIR / "acp740_dataset_with_features.csv"
RANDOM_STATE = 42





## Standardization and statistical feature selection

In [ ]:
combined_dataset = pd.read_csv(DATA_PATH)

features_to_scale = combined_dataset.drop(columns=["data", "label"])
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_to_scale)
scaled_df = pd.DataFrame(scaled_features, columns=features_to_scale.columns)
scaled_df["data"] = combined_dataset["data"].values
scaled_df["label"] = combined_dataset["label"].values
combined_dataset = scaled_df[["data", "label", *features_to_scale.columns]]

group_0 = combined_dataset.loc[combined_dataset["label"] == 0].drop(columns=["data", "label"])
group_1 = combined_dataset.loc[combined_dataset["label"] == 1].drop(columns=["data", "label"])
t_stats, p_values = ttest_ind(group_0, group_1, equal_var=False)
ttest_results = pd.DataFrame({"feature": group_0.columns, "t_stat": t_stats, "p_value": p_values})

nan_features = ttest_results.loc[ttest_results["p_value"].isna(), "feature"]
cols_to_drop = [column for column in nan_features if column in combined_dataset.columns]
combined_dataset = combined_dataset.drop(columns=cols_to_drop)
ttest_results = ttest_results.dropna()

p_values_df = ttest_results[["feature", "p_value"]].copy()
_, q_vals, _, _ = multipletests(pvals=p_values_df["p_value"], method="fdr_bh")
p_values_df["q_value"] = q_vals
significant_features = p_values_df.loc[p_values_df["q_value"] < 0.01, "feature"].tolist()
selected_columns = ["data", "label", *significant_features]
filtered_dataset = combined_dataset[selected_columns]
cleaned_data = filtered_dataset


## Feature groups

In [ ]:
feature_group_config = {
    "aac": "features_aac.csv",
    "dipeptide": "features_dipeptide.csv",
    "geary_autocorrelation": "features_geary_autocorrelation.csv",
    "moran_autocorrelation": "features_moran_autocorrelation.csv",
    "normalized_moreau_broto_autocor": "features_normalized_moreau_broto_autocor.csv",
    "physico_chemical_composition": "features_physico_chemical_composition.csv",
    "physico_chemical_distribution": "features_physico_chemical_distribution.csv",
    "physico_chemical_transition": "features_physico_chemical_transition.csv",
    "physicochemical": "features_physicochemical.csv",
    "pseudo_amino_acid_composition": "features_pseudo_amino_acid_composition.csv",
    "quasi_sequence": "features_quasi.csv",
    "soc": "features_soc.csv",
}

feature_frames = {}
feature_column_lists = {}
for group_name, filename in feature_group_config.items():
    source_columns = pd.read_csv(DATA_DIR / filename).columns.tolist()
    valid_columns = [column for column in source_columns if column in cleaned_data.columns]
    feature_frames[group_name] = cleaned_data[valid_columns]
    feature_column_lists[group_name] = valid_columns

# Keep the original feature-group variables for downstream use.
aac_features_df = feature_frames["aac"]
features_dipeptide_df = feature_frames["dipeptide"]
features_geary_autocorrelation_df = feature_frames["geary_autocorrelation"]
features_moran_autocorrelation_df = feature_frames["moran_autocorrelation"]
features_normalized_moreau_broto_autocor_df = feature_frames["normalized_moreau_broto_autocor"]
features_physico_chemical_composition_df = feature_frames["physico_chemical_composition"]
features_physico_chemical_distribution_df = feature_frames["physico_chemical_distribution"]
features_physico_chemical_transition_df = feature_frames["physico_chemical_transition"]
features_physicochemical_df = feature_frames["physicochemical"]
features_pseudo_amino_acid_composition_df = feature_frames["pseudo_amino_acid_composition"]
features_quasi_sequence_df = feature_frames["quasi_sequence"]
features_soc_df = feature_frames["soc"]


## Group-level statistics and feature selection

In [ ]:
group_statistics = []
for group_name in feature_group_config:
    matching_rows = ttest_results[ttest_results["feature"].isin(feature_column_lists[group_name])]
    sum_t_stat = matching_rows["t_stat"].sum()
    feature_count = len(matching_rows)
    z_value = sum_t_stat / math.sqrt(feature_count) if feature_count > 0 else 0
    p_value = 2 * (1 - norm.cdf(abs(z_value)))
    group_statistics.append({"group": group_name, "p_value": p_value})

groups_stat = pd.DataFrame(group_statistics, columns=["group", "p_value"])
_, q_values, _, _ = multipletests(groups_stat["p_value"].values, method="fdr_bh")
groups_stat["q_value"] = q_values
significant_groups = groups_stat[groups_stat["q_value"] < 0.01]

# Use the feature groups selected for the ACP740 dataset.
selected_group_names = [
    "physico_chemical_composition",
    "physico_chemical_distribution",
    "quasi_sequence",
    "physicochemical",
    "dipeptide",
    "geary_autocorrelation", 
]


selected_group_columns = [
    column
    for group_name in selected_group_names
    for column in feature_column_lists[group_name]
]
selected_groups_columns = ["data", "label", *selected_group_columns]
new_cleaned_data = cleaned_data.loc[:, ~cleaned_data.columns.duplicated()]
new_cleaned_data = new_cleaned_data[[column for column in cleaned_data.columns if column in selected_groups_columns]]


## SVM

In [ ]:
X = new_cleaned_data.drop(columns=["data", "label"])
y = new_cleaned_data["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    shrinking=True,
    random_state=RANDOM_STATE,
)
scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "mcc": make_scorer(matthews_corrcoef),
}
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    model, X_train, y_train, cv=cv, scoring=scoring, return_train_score=False, n_jobs=-1
)

def mean_std(scores):
    return f"{np.mean(scores):.4f} +/- {np.std(scores):.4f}"

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print("\n10-Fold CV Results (Train Set - SVM):")
print(f"Accuracy     : {mean_std(cv_results['test_accuracy'])}")
print(f"F1 Score     : {mean_std(cv_results['test_f1'])}")
print(f"Sensitivity  : {mean_std(cv_results['test_recall'])}")
print(f"Precision    : {mean_std(cv_results['test_precision'])}")
print(f"AUC (ROC)    : {mean_std(cv_results['test_roc_auc'])}")
print(f"MCC          : {mean_std(cv_results['test_mcc'])}")

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("\nFinal Test Set Results:")
print(f"Accuracy     : {(tp + tn) / (tp + tn + fp + fn):.4f}")
print(f"F1 Score     : {2 * tp / (2 * tp + fp + fn):.4f}")
print(f"Sensitivity  : {tp / (tp + fn):.4f}")
print(f"Specificity  : {tn / (tn + fp):.4f}")
print(f"Precision    : {tp / (tp + fp):.4f}")
print(f"AUC (ROC)    : {roc_auc_score(y_test, y_pred):.4f}")
print(f"MCC          : {matthews_corrcoef(y_test, y_pred):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")
